In [2]:

# Cell 1 - inspect the environment (read-only)
# 1) Where Kaggle mounted our 4 datasets, and the files inside each
# 2) Which GPU we got and how much memory it has
# 3) How much free disk space /kaggle/working and /tmp have (we must unzip ~17 GB)

import subprocess

print("=== INPUTS ===")
print(subprocess.run("ls -la /kaggle/input/*", shell=True, capture_output=True, text=True).stdout)

print("=== GPU ===")
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)

print("=== DISK ===")
print(subprocess.run("df -h /kaggle/working /tmp", shell=True, capture_output=True, text=True).stdout)

=== INPUTS ===
total 16
drwxr-xr-x 4 root root 4096 Nov  8 00:36 .
drwxr-xr-x 3 root root 4096 Nov  8 00:36 ..
drwxr-xr-x 3 root root 4096 Nov  8 00:36 harsh2035
drwxr-xr-x 6 root root 4096 Nov  8 00:36 sakshampainuly007

=== GPU ===
Sun Nov  8 00:36:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 

In [3]:
# Cell 2 - inspect the attached datasets (read-only)
# For each dataset mounted under /kaggle/input/<owner>/<dataset>/ we print
# at most 15 entries, so the output stays small even if a dataset has
# thousands of files. We also count the files in each one.

import os

INPUT_ROOT = "/kaggle/input"

for owner in sorted(os.listdir(INPUT_ROOT)):
    owner_dir = os.path.join(INPUT_ROOT, owner)
    for ds in sorted(os.listdir(owner_dir)):
        ds_dir = os.path.join(owner_dir, ds)
        n_files = sum(len(files) for _, _, files in os.walk(ds_dir))
        print(f"=== {owner}/{ds}   ({n_files} files in total)")
        for i, name in enumerate(sorted(os.listdir(ds_dir))):
            if i >= 15:
                print("    ... (more entries not shown)")
                break
            path = os.path.join(ds_dir, name)
            if os.path.isfile(path):
                print(f"    {name}   {os.path.getsize(path):,} bytes")
            else:
                print(f"    {name}/")
        print()

=== datasets/harsh2035   (11352 files in total)
    model-a/

=== datasets/sakshampainuly007   (4 files in total)
    uwct-biology-raw/
    uwct-harsh-model-a-2026-11/
    uwct-harsh-v1-colour-2026-11/
    uwct-harsh-v1-extra-2026-11/



In [5]:
# Cell 3 - locate our 4 uploaded archives and check their sizes (read-only)
# Expected sizes are the byte counts of the verified local copies (2026-11-06).
# A size match is a quick first check; the sha256 check comes in the next cell.

import os

OUR_ROOT = "/kaggle/input/datasets/sakshampainuly007"
EXPECTED_BYTES = {
    "biology.zip.bin":                5_478_527_588,
    "harsh_model_a_2026-11.zip.bin": 10_121_406_479,
    "harsh_v1_extra_2026-11.zip.bin":  1_819_177_935,
    "harsh_v1_colour_2026-11.zip.bin":   263_128_178,
}

found = {}
for dirpath, _, files in os.walk(OUR_ROOT):
    for f in files:
        found[f] = os.path.join(dirpath, f)

for name, expected in EXPECTED_BYTES.items():
    if name not in found:
        print(f"MISSING   {name}")
        continue
    size = os.path.getsize(found[name])
    status = "OK      " if size == expected else "MISMATCH"
    print(f"{status}  {found[name]}")
    print(f"          size {size:,} B (expected {expected:,} B)")

extra = set(found) - set(EXPECTED_BYTES)
if extra:
    print("Unexpected files:", sorted(extra))

OK        /kaggle/input/datasets/sakshampainuly007/uwct-biology-raw/biology.zip.bin
          size 5,478,527,588 B (expected 5,478,527,588 B)
OK        /kaggle/input/datasets/sakshampainuly007/uwct-harsh-model-a-2026-11/harsh_model_a_2026-11.zip.bin
          size 10,121,406,479 B (expected 10,121,406,479 B)
OK        /kaggle/input/datasets/sakshampainuly007/uwct-harsh-v1-extra-2026-11/harsh_v1_extra_2026-11.zip.bin
          size 1,819,177,935 B (expected 1,819,177,935 B)
OK        /kaggle/input/datasets/sakshampainuly007/uwct-harsh-v1-colour-2026-11/harsh_v1_colour_2026-11.zip.bin
          size 263,128,178 B (expected 263,128,178 B)


In [6]:
# Cell 4 - verify archive integrity with sha256
# Expected hashes were computed on the verified local copies before upload
# (D:\uwct_kaggle_upload\*\*.sha256). All 4 must print MATCH before we unpack.
# The result is also saved to /kaggle/working/uwct_logs/archive_sha256.txt,
# which is kept when the notebook version is saved.

import hashlib, os, time

ARCHIVES = {
    "/kaggle/input/datasets/sakshampainuly007/uwct-biology-raw/biology.zip.bin":
        "88d3ba76e3e9ed55af7929667e8e3abbfb51f6d7444130c27627cbc3494899b3",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-model-a-2026-11/harsh_model_a_2026-11.zip.bin":
        "ecbcb2a58403365b79c6764d714d414ca97a0e4c80d088db43518f76b76dc2c3",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-v1-extra-2026-11/harsh_v1_extra_2026-11.zip.bin":
        "02ae7b10f23a6f5c42b8598039b3a74c7e0ded47d3bcea5d124f1b3e4e606cbd",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-v1-colour-2026-11/harsh_v1_colour_2026-11.zip.bin":
        "405c373d1d3680b98433ab537c83e73fd8a961a607ae616415d6cfaf2c1852e7",
}

LOG_DIR = "/kaggle/working/uwct_logs"
os.makedirs(LOG_DIR, exist_ok=True)

def sha256_of(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

lines, all_ok = [], True
for path, expected in ARCHIVES.items():
    t0 = time.time()
    got = sha256_of(path)
    ok = got == expected
    all_ok &= ok
    line = f"{'MATCH   ' if ok else 'MISMATCH'}  {os.path.basename(path)}  {got}  ({time.time() - t0:.0f} s)"
    print(line)
    lines.append(line)

summary = "ALL 4 ARCHIVES VERIFIED" if all_ok else "STOP: at least one archive does not match"
print("\n" + summary)
with open(os.path.join(LOG_DIR, "archive_sha256.txt"), "w") as f:
    f.write("\n".join(lines + ["", summary]) + "\n")

MATCH     biology.zip.bin  88d3ba76e3e9ed55af7929667e8e3abbfb51f6d7444130c27627cbc3494899b3  (39 s)
MATCH     harsh_model_a_2026-11.zip.bin  ecbcb2a58403365b79c6764d714d414ca97a0e4c80d088db43518f76b76dc2c3  (61 s)
MATCH     harsh_v1_extra_2026-11.zip.bin  02ae7b10f23a6f5c42b8598039b3a74c7e0ded47d3bcea5d124f1b3e4e606cbd  (14 s)
MATCH     harsh_v1_colour_2026-11.zip.bin  405c373d1d3680b98433ab537c83e73fd8a961a607ae616415d6cfaf2c1852e7  (4 s)

ALL 4 ARCHIVES VERIFIED


In [7]:
# Cell 5 - unpack the verified archives to /tmp/uwct_raw (raw data, treated read-only)
# /tmp is used because /kaggle/working is only 20 GB; /tmp is wiped when the
# session ends, so this cell must be re-run in every new session.
# Each archive has one top folder named like our repo's data/raw/<folder>.
# Safety: every entry is checked BEFORE extraction (no absolute paths, no "..",
# correct top folder). Nothing is overwritten: an existing target stops the cell.

import os, zipfile, shutil, time

RAW_DIR = "/tmp/uwct_raw"
IN = "/kaggle/input/datasets/sakshampainuly007"
ARCHIVES = {   # archive path -> expected top folder
    f"{IN}/uwct-biology-raw/biology.zip.bin":                              "biology",
    f"{IN}/uwct-harsh-model-a-2026-11/harsh_model_a_2026-11.zip.bin":      "harsh_model_a_2026-11",
    f"{IN}/uwct-harsh-v1-extra-2026-11/harsh_v1_extra_2026-11.zip.bin":    "harsh_v1_extra_2026-11",
    f"{IN}/uwct-harsh-v1-colour-2026-11/harsh_v1_colour_2026-11.zip.bin":  "harsh_v1_colour_2026-11",
}

os.makedirs(RAW_DIR, exist_ok=True)
log = []
for archive, top in ARCHIVES.items():
    target = os.path.join(RAW_DIR, top)
    if os.path.exists(target):
        raise SystemExit(f"STOP: {target} already exists - not overwriting. "
                         f"Restart the session or remove it deliberately.")
    t0 = time.time()
    with zipfile.ZipFile(archive) as zf:
        members = zf.infolist()
        bad = [m.filename for m in members
               if m.filename.startswith("/") or ".." in m.filename.split("/")
               or m.filename.split("/")[0] != top]
        if bad:
            raise SystemExit(f"STOP: unsafe or unexpected entries in {archive}: {bad[:5]}")
        zf.extractall(RAW_DIR)
    n_files = sum(1 for m in members if not m.is_dir())
    line = f"{top:26s} {n_files:6d} files extracted   ({time.time() - t0:.0f} s)"
    print(line)
    log.append(line)

total, used, free = shutil.disk_usage(RAW_DIR)
disk = f"/tmp free after extraction: {free / 1e9:.0f} GB"
print("\n" + disk)
with open("/kaggle/working/uwct_logs/extraction.txt", "w") as f:
    f.write("\n".join(log + ["", disk]) + "\n")

biology                      4407 files extracted   (26 s)
harsh_model_a_2026-11       11359 files extracted   (76 s)
harsh_v1_extra_2026-11       1375 files extracted   (15 s)
harsh_v1_colour_2026-11       677 files extracted   (3 s)

/tmp free after extraction: 1020 GB


In [8]:
# Cell 6 - clone the project repository (private) at the Model A branch
# The token is read from Kaggle Secrets and passed as a one-off HTTP header,
# so it is NOT written into the repo's .git/config and never printed.
# The exact commit used is printed and logged, so every result in this
# notebook can be traced back to the code that produced it.

import base64, os, subprocess
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/SakshamPainuly/underwater-creature-tracking.git"
BRANCH = "p7/model-a-plan"
REPO_DIR = "/tmp/uwct_repo"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()

if os.path.exists(REPO_DIR):
    raise SystemExit(f"STOP: {REPO_DIR} already exists - not overwriting.")

r = subprocess.run(
    ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}",
     "clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO_DIR],
    capture_output=True, text=True)
del token, auth
if r.returncode != 0:
    # print git's message with any secret material removed
    raise SystemExit("STOP: clone failed:\n" + r.stderr.replace(REPO_URL, "<repo>"))

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args],
                          capture_output=True, text=True).stdout.strip()

info = [
    f"repo    : {REPO_URL}",
    f"branch  : {git('rev-parse', '--abbrev-ref', 'HEAD')}",
    f"commit  : {git('rev-parse', 'HEAD')}",
    f"message : {git('log', '-1', '--format=%s (%ad)', '--date=short')}",
    f"dirty   : {'yes' if git('status', '--porcelain') else 'no'}",
]
print("\n".join(info))
with open("/kaggle/working/uwct_logs/code_version.txt", "w") as f:
    f.write("\n".join(info) + "\n")

repo    : https://github.com/SakshamPainuly/underwater-creature-tracking.git
branch  : p7/model-a-plan
commit  : 98d8208cfd33acddaf4052bba5543b0ecf1c89bd
message : D-031 accepted (2026-11-04) (2026-11-04)
dirty   : no


In [9]:
# Cell 7 - raw data check (CLAUDE.md section 4): extracted files vs committed fingerprints
#   biology : structural fingerprint (D-011)  -> paths + sizes (+ txt/xlsx/docx hashes)
#   Harsh x3: full fingerprints (D-031)        -> sha256 of every file (--full)
# Every check must end with "RAW DATA MATCHES FINGERPRINT".

import subprocess, sys

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
FP = f"{REPO}/data/processed"
CHECKS = [
    ("biology",                 f"{FP}/raw_fingerprint_d011_post_cleanup.json",       []),
    ("harsh_model_a_2026-11",   f"{FP}/raw_fingerprint_harsh_model_a_2026-11.json",   ["--full"]),
    ("harsh_v1_extra_2026-11",  f"{FP}/raw_fingerprint_harsh_v1_extra_2026-11.json",  ["--full"]),
    ("harsh_v1_colour_2026-11", f"{FP}/raw_fingerprint_harsh_v1_colour_2026-11.json", ["--full"]),
]

report, all_ok = [], True
for folder, fingerprint, extra in CHECKS:
    r = subprocess.run([sys.executable, f"{REPO}/scripts/check_raw_unchanged.py",
                        "--root", f"{RAW}/{folder}", "--fingerprint", fingerprint, *extra],
                       capture_output=True, text=True)
    all_ok &= r.returncode == 0
    block = f"===== {folder}\n{r.stdout.strip()}" + (f"\n{r.stderr.strip()}" if r.stderr.strip() else "")
    print(block + "\n")
    report.append(block)

summary = "ALL 4 RAW CHECKS PASSED" if all_ok else "STOP: a raw check failed"
print(summary)
with open("/kaggle/working/uwct_logs/raw_check.txt", "w") as f:
    f.write("\n\n".join(report) + f"\n\n{summary}\n")


===== biology
check: structural (paths+sizes; txt/xlsx/docx hashed); subset: all; root: /tmp/uwct_raw/biology
fingerprint: /tmp/uwct_repo/data/processed/raw_fingerprint_d011_post_cleanup.json
files expected: 4407  present: 4407
  missing: 0
  added: 0
  size changed: 0
  content changed: 0
RAW DATA MATCHES FINGERPRINT

===== harsh_model_a_2026-11
check: full content (cryptographic); subset: all; root: /tmp/uwct_raw/harsh_model_a_2026-11
fingerprint: /tmp/uwct_repo/data/processed/raw_fingerprint_harsh_model_a_2026-11.json
files expected: 11359  present: 11359
  missing: 0
  added: 0
  size changed: 0
  content changed: 0
RAW DATA MATCHES FINGERPRINT

===== harsh_v1_extra_2026-11
check: full content (cryptographic); subset: all; root: /tmp/uwct_raw/harsh_v1_extra_2026-11
fingerprint: /tmp/uwct_repo/data/processed/raw_fingerprint_harsh_v1_extra_2026-11.json
files expected: 1375  present: 1375
  missing: 0
  added: 0
  size changed: 0
  content changed: 0
RAW DATA MATCHES FINGERPRINT

====

In [10]:
# Cell 8 - install the pinned training package and record the environment (CLAUDE.md section 6)
# requirements-gpu.txt pins ultralytics==8.4.155 and says: do not reinstall torch.
# We print torch's version before and after to prove the install left it untouched.
# The environment report is saved to /kaggle/working/uwct_logs/env_report.json.

import subprocess, sys, json

REPO = "/tmp/uwct_repo"
torch_ver = lambda: subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                                   capture_output=True, text=True).stdout.strip()

before = torch_ver()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-gpu.txt"],
                   capture_output=True, text=True)
print("pip exit code:", r.returncode)
tail = (r.stdout + r.stderr).strip().splitlines()[-5:]
if tail:
    print("pip (last lines):\n  " + "\n  ".join(tail))
after = torch_ver()
print(f"torch before: {before}   after: {after}   {'UNCHANGED' if before == after else 'CHANGED - tell Claude'}")

r = subprocess.run([sys.executable, f"{REPO}/scripts/env_report.py",
                    "--out", "/kaggle/working/uwct_logs/env_report.json"],
                   capture_output=True, text=True)
print(r.stderr.strip() or "")
print(json.dumps(json.load(open("/kaggle/working/uwct_logs/env_report.json")), indent=1))

pip exit code: 0
torch before: 2.11.0+cu130   after: 2.11.0+cu130   UNCHANGED

{
 "timestamp_utc": "2026-11-08T01:00:10+00:00",
 "os": "Linux-6.18.55+-x86_64-with-glibc2.39",
 "python": "3.13.16",
 "cpu_count": 4,
 "torch": "2.11.0+cu130",
 "cuda_available": true,
 "cuda_version": "13.0",
 "cudnn_version": 92700,
 "gpus": [
  {
   "index": 0,
   "name": "Tesla T4",
   "vram_gb": 14.56
  },
  {
   "index": 1,
   "name": "Tesla T4",
   "vram_gb": 14.56
  }
 ],
 "ultralytics": "8.4.155",
 "nvidia_smi": [
  {
   "name": "Tesla T4",
   "memory_total": "15360 MiB",
   "driver_version": "580.178.04"
  },
  {
   "name": "Tesla T4",
   "memory_total": "15360 MiB",
   "driver_version": "580.178.04"
  }
 ]
}


In [11]:
# Cell 9 - update the cloned repo to branch p8/model-a-e007 (v4 builder, frozen v4, E007 config)
# Same token handling as Cell 6: one-off header, never stored or printed.
# The new commit is logged, replacing the old code_version.txt.

import base64, os, subprocess
from kaggle_secrets import UserSecretsClient

REPO_DIR = "/tmp/uwct_repo"
BRANCH = "p8/model-a-e007"
for needed in (REPO_DIR, "/tmp/uwct_raw/biology"):
    if not os.path.isdir(needed):
        raise SystemExit(f"STOP: {needed} is missing - the session restarted; re-run Cells 1-8 "
                         f"(with BRANCH = '{BRANCH}' in Cell 6).")

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
r = subprocess.run(["git", "-C", REPO_DIR, "-c", f"http.extraHeader=Authorization: Basic {auth}",
                    "fetch", "--quiet", "origin", f"{BRANCH}:{BRANCH}"], capture_output=True, text=True)
del token, auth
if r.returncode != 0:
    raise SystemExit("STOP: fetch failed:\n" + r.stderr)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", BRANCH], check=True)

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args], capture_output=True, text=True).stdout.strip()

info = [
    f"branch  : {git('rev-parse', '--abbrev-ref', 'HEAD')}",
    f"commit  : {git('rev-parse', 'HEAD')}",
    f"message : {git('log', '-1', '--format=%s (%ad)', '--date=short')}",
    f"dirty   : {'yes' if git('status', '--porcelain') else 'no'}",
]
print("\n".join(info))
with open("/kaggle/working/uwct_logs/code_version.txt", "w") as f:
    f.write("\n".join(info) + "\n")

branch  : p8/model-a-e007
commit  : 428f5bde7fd957b53821a900b362231a09ca2564
message : D-032: v4 frozen, E007 config (all safe Model A training data) (2026-11-07)
dirty   : no


In [12]:
# Cell 10 - build the YOLO view of frozen dataset v4 (D-032)
# Raw roots point at the verified extracted data in /tmp/uwct_raw (read-only by the builder).
# Output goes to /tmp (not /kaggle/working: ~6 GB of image copies, rebuilt each session).
# The fingerprint must equal the one committed in the E007 config.

import json, os, subprocess, sys, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
OUT = "/tmp/uwct_yolo/lobster_mixed_v4"
EXPECTED_FP = "bbd4a8b172fd590b18d2e8c26be05c85cb3c3246a2e60a3b3513590b931cd13f"

env = dict(os.environ,
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")
t0 = time.time()
r = subprocess.run([sys.executable, f"{REPO}/scripts/prepare_yolo_dataset.py", "--version", "v4", "--out", OUT],
                   capture_output=True, text=True, env=env)
if r.returncode != 0:
    raise SystemExit("STOP: build failed:\n" + (r.stdout + r.stderr)[-3000:])

s = json.load(open(f"{OUT}/build_summary.json"))
print(f"built in {time.time() - t0:.0f} s   version {s['dataset_version']} ({s['version_status']})   "
      f"git {s['git']['commit'][:7]} dirty={s['git']['dirty']}")
for split, c in {**s["splits"], **s["eval_only_splits"]}.items():
    print(f"  {split:15s} images {c['images']:5d}   with lobster {c['positive']:5d}   "
          f"empty {c['negative']:4d}   boxes {c['instances']:6d}")
print("  not built:", sum(s["not_included"].values()), s["not_included"])
ok = s["dataset_fingerprint"] == EXPECTED_FP
print(f"\nfingerprint {s['dataset_fingerprint']}\n{'MATCHES the E007 config' if ok else 'STOP: differs from the E007 config'}")

os.system(f"cp {OUT}/build_summary.json /kaggle/working/uwct_logs/build_summary_v4.json")

built in 79 s   version v4 (frozen)   git 428f5bd dirty=False
  train           images  4111   with lobster  3721   empty  390   boxes  11784
  val             images   324   with lobster   269   empty   55   boxes   1010
  test            images   348   with lobster   321   empty   27   boxes   1512
  test_zenodo_sk  images   428   with lobster   143   empty  285   boxes    143
  not built: 1202 {'ancestor_label_excluded': 65, 'ancestor_not_train': 992, 'label_review_excluded': 90, 'label_review_low_confidence_empty': 31, 'zenodo_no_site': 24}

fingerprint bbd4a8b172fd590b18d2e8c26be05c85cb3c3246a2e60a3b3513590b931cd13f
MATCHES the E007 config


0

In [13]:
# Cell 11 - Ultralytics reads the built dataset (CLAUDE.md section 6: must print VERIFIED)
# Expected time: about 1-2 min (Ultralytics scans ~5,200 images and labels once).
# Checks per split that Ultralytics sees the same images / empty frames / boxes as build_summary.json,
# including the held-out Zenodo sk split (its own data file).

import subprocess, sys, time

REPO = "/tmp/uwct_repo"
OUT = "/tmp/uwct_yolo/lobster_mixed_v4"

t0 = time.time()
r = subprocess.run([sys.executable, f"{REPO}/scripts/verify_yolo_dataset.py", "--dataset", OUT],
                   capture_output=True, text=True)
print(r.stdout.strip())
print(f"\n(exit code {r.returncode}, {time.time() - t0:.0f} s)")
if r.returncode != 0:
    print("stderr (last lines):\n" + "\n".join(r.stderr.strip().splitlines()[-15:]))
with open("/kaggle/working/uwct_logs/verify_yolo_v4.txt", "w") as f:
    f.write(r.stdout)

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 882.2±784.0 MB/s, size: 1816.2 KB)

val: Scanning /tmp/uwct_yolo/lobster_mixed_v4/labels/train... 8 images, 3 backgrounds, 0 corrupt: 0% ──────────── 8/4111 23.4it/s 0.1s<2:55
val: Scanning /tmp/uwct_yolo/lobster_mixed_v4/labels/train... 19 images, 3 backgrounds, 0 corrupt: 0% ──────────── 19/4111 45.5it/s 0.2s<1:30
val: Scanning /tmp/uwct_yolo/lobster_mixed_v4/labels/train... 33 images, 3 backgrounds, 0 corrupt: 0% ──────────── 33/4111 70.1it/s 0.3s<58.1s
val: Scanning /tmp/uwct_yolo/lobster_mixed_v4/labels/train... 43 images, 3 backgrounds, 0 corrupt: 1% ──────────── 43/4111 77.6it/s 0.4s<52.4s
val: Scanning /tmp/uwct_yolo/lobster_mixed_v4/labels/train... 51 images, 3 backgrounds, 0 corrupt: 1% ──────────── 51/4111 75.3it/s 0.5s<53.9s
val: Scanning /tmp/uwct_yolo/lobster_mixed_v4/labels/train... 59 images, 3 backgrounds, 0 corrupt: 1% ──────────── 59/4111 72.5it/s 0.7s<55.9s
val: Scanning /tmp/uwct_yolo/lobster_mixed_v4/labels/train..

In [14]:
# Cell 12 - leakage check (D-031 point 4, rules 1-6) on the raw data in /tmp/uwct_raw
# Expected time: about 3-5 min (sha256 + dHash of every train/val/test image).
# Rules 1-5: split and ancestry rules from the CSVs. Rule 6: identical files and near-duplicates
# (dHash distance <= 4) across train / val / test.
# Known result (D-031, reviewed 2026-11-04): 2 near-duplicate flags, both pairing the Zenodo mpa frame
# cam2_120822_rec_001_frame_0021 (and its copy-paste image) with sk test frame cam3_120824_rec087_frame_0038;
# a visual check found different scenes, so all images are kept. Any OTHER error stops us.

import subprocess, sys, time

REPO = "/tmp/uwct_repo"
V4 = f"{REPO}/data/processed/v4"
REVIEWED = ("mpa_cam2_120822_rec_001_frame_0021", "sk_cam3_120824_rec087_frame_0038")

t0 = time.time()
r = subprocess.run([sys.executable, f"{REPO}/scripts/check_leakage.py",
                    "--manifest", f"{V4}/mixed_manifest.csv", "--groups", f"{V4}/zenodo_groups.csv",
                    "--image-root", "/tmp/uwct_raw", "--show", "50"],
                   capture_output=True, text=True)
out = r.stdout.strip()
print(out)
print(f"\n(exit code {r.returncode}, {time.time() - t0:.0f} s)")
if r.stderr.strip():
    print("stderr (last lines):\n" + "\n".join(r.stderr.strip().splitlines()[-10:]))

errors = [l.strip() for l in out.splitlines() if l.startswith("  ") and ("duplicate" in l or "rule" in l.lower())]
known = [e for e in errors if e.startswith("near-duplicate") and all(k in e for k in REVIEWED)]
if r.returncode == 0:
    verdict = "LEAKAGE: PASSED (no flags)"
elif "LEAKAGE CHECK: FAILED (2 error(s))" in out and len(known) == 2:
    verdict = "LEAKAGE: PASSED - the only 2 flags are the pair reviewed in D-031 (different scenes)"
else:
    verdict = "STOP: unexpected leakage errors - paste this output to Claude"
print("\n" + verdict)
with open("/kaggle/working/uwct_logs/leakage_check.txt", "w") as f:
    f.write(out + "\n\n" + verdict + "\n")

hashing 5211 images ...
{
  "excluded": {
    "ours/colour_change": 6,
    "ours/copy_paste": 257,
    "ours/diffusion": 494,
    "ours/original": 121,
    "zenodo/colour_change": 107,
    "zenodo/copy_paste": 193,
    "zenodo/original": 24
  },
  "test": {
    "ours/original": 348,
    "zenodo/original": 428
  },
  "train": {
    "ours/colour_change": 192,
    "ours/copy_paste": 516,
    "ours/diffusion": 1312,
    "ours/original": 1312,
    "zenodo/colour_change": 93,
    "zenodo/copy_paste": 206,
    "zenodo/original": 480
  },
  "val": {
    "ours/original": 324
  }
}
  near-duplicate (dHash distance 4): zenodo/copy_paste/paste_0279__kso__mpa_cam2_120822_rec_001_frame_0021_jpeg.rf.5501ceb3b78bd8e1c9cc02a6575fd5f7.jpg (train) ~ zenodo/original/kso__sk_cam3_120824_rec087_frame_0038_jpeg.rf.78529232cb9dd39203ccc04e8040d210.jpg (test)
  near-duplicate (dHash distance 4): zenodo/original/kso__mpa_cam2_120822_rec_001_frame_0021_jpeg.rf.5501ceb3b78bd8e1c9cc02a6575fd5f7.jpg (train) ~ zenod

In [15]:
# Cell 13 - smoke test (CLAUDE.md section 6): 1 epoch on 10% of train, validated on val
# Expected time: about 4-8 min (first Kaggle run: includes the yolo26s.pt download).
# Runs every training-side check (config, clean git commit, frozen v4 view + fingerprint,
# Ultralytics label scan, pretrained checkpoint) and then trains briefly. The full Ultralytics
# output goes to /kaggle/working/uwct_logs/smoke_E007.log; only a summary is printed here.

import csv, glob, json, os, subprocess, sys, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
RUNS = "/kaggle/working/uwct_runs"        # kept when the notebook version is saved
WEIGHTS = "/kaggle/working/uwct_weights"  # pretrained checkpoint download
LOG = "/kaggle/working/uwct_logs/smoke_E007.log"
os.makedirs(RUNS, exist_ok=True); os.makedirs(WEIGHTS, exist_ok=True)

env = dict(os.environ,
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")
cmd = [sys.executable, f"{REPO}/scripts/train_yolo.py",
       "--config", f"{REPO}/configs/experiments/E007_yolo_model_a_all.yaml",
       "--runs-root", RUNS, "--weights-dir", WEIGHTS,
       "--dataset", "/tmp/uwct_yolo/lobster_mixed_v4", "--smoke"]

t0 = time.time()
with open(LOG, "w") as log:
    r = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, text=True, env=env)
minutes = (time.time() - t0) / 60
lines = open(LOG).read().splitlines()
print(f"exit code {r.returncode}   total {minutes:.1f} min   (full log: {LOG}, {len(lines)} lines)")

if r.returncode != 0:
    print("\nLast 25 log lines:")
    print("\n".join(lines[-25:]))
else:
    run_dir = sorted(glob.glob(f"{RUNS}/E007_smoke_*"))[-1]
    rec = json.load(open(f"{run_dir}/run_record.json"))
    print(f"run folder : {run_dir}")
    print(f"git commit : {rec['git']['commit'][:7]}   dataset {rec['dataset']['version']} "
          f"fingerprint {rec['dataset']['fingerprint'][:12]}...")
    print(f"model      : {rec['model']['name']}  sha256 {rec['model']['checkpoint_sha256'][:12]}...")
    print(f"val metrics (1 epoch on 10% - pipeline check only, NOT a result): {rec['val_metrics']}")
    with open(f"{run_dir}/train/results.csv") as f:
        row = list(csv.DictReader(f))[-1]
    print(f"epoch time (10% of train + val): {float(row['time']):.0f} s")


exit code 0   total 2.3 min   (full log: /kaggle/working/uwct_logs/smoke_E007.log, 570 lines)
run folder : /kaggle/working/uwct_runs/E007_smoke_20261108T014155
git commit : 428f5bd   dataset v4 fingerprint bbd4a8b172fd...
model      : yolo26s.pt  sha256 646f8bc3fe0a...
val metrics (1 epoch on 10% - pipeline check only, NOT a result): {'split': 'val', 'weights': 'best.pt', 'precision': 0.0861461379962942, 'recall': 0.12376237623762376, 'map50': 0.033286260827030505, 'map50_95': 0.013111686768084493, 'ultralytics_results_dict': {'metrics/precision(B)': 0.0861461379962942, 'metrics/recall(B)': 0.12376237623762376, 'metrics/mAP50(B)': 0.033286260827030505, 'metrics/mAP50-95(B)': 0.013111686768084493, 'fitness': 0.013111686768084493}, 'note': 'Ultralytics val defaults (conf=0.001, iou=0.7); operating threshold not chosen yet.'}
epoch time (10% of train + val): 25 s


In [16]:
# Cell 14 - E007 full training (D-032): all 4,111 safe v4 train images, E001 settings, val = 324 v1 frames
# Expected time: about 3.5-5 h for 100 epochs (smoke test: ~2-3 min per full epoch on one T4);
# early stopping (patience 30) can end it sooner.
# The test splits are never used: training gets a data file with train + val only.
# Full Ultralytics output -> /kaggle/working/uwct_logs/train_E007.log
# Printed here: one line per epoch (losses, val P/R/mAP, best so far, time), then the run record.
# Checkpoints: /kaggle/working/uwct_runs/E007/train/weights/{last,best}.pt (kept with the version).

import csv, json, os, subprocess, sys, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
RUNS = "/kaggle/working/uwct_runs"
RUN = f"{RUNS}/E007"
LOG = "/kaggle/working/uwct_logs/train_E007.log"
EPOCHS = 100

env = dict(os.environ,
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")
cmd = [sys.executable, f"{REPO}/scripts/train_yolo.py",
       "--config", f"{REPO}/configs/experiments/E007_yolo_model_a_all.yaml",
       "--runs-root", RUNS, "--weights-dir", "/kaggle/working/uwct_weights",
       "--dataset", "/tmp/uwct_yolo/lobster_mixed_v4"]

def f(row, *keys):
    for k in keys:
        if row.get(k, "").strip():
            return float(row[k])
    return float("nan")

log = open(LOG, "w")
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, text=True, env=env)
print(f"E007 started {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}; one line per finished epoch\n")
print(" epoch | ep time | box loss  cls loss | val P   val R   mAP50  mAP50-95 | best mAP50 (epoch) | elapsed")

shown, best, prev_t = 0, (-1.0, 0), 0.0
while True:
    done = proc.poll() is not None
    results = f"{RUN}/train/results.csv"
    if os.path.exists(results):
        with open(results) as fh:
            rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(fh)]
        for row in rows[shown:]:
            ep = int(f(row, "epoch"))
            t = f(row, "time")
            m50 = f(row, "metrics/mAP50(B)")
            if m50 > best[0]:
                best = (m50, ep)
            print(f" {ep:3d}/{EPOCHS} | {(t - prev_t) / 60:5.1f} m | {f(row, 'train/box_loss'):7.3f}  {f(row, 'train/cls_loss'):7.3f}  | "
                  f"{f(row, 'metrics/precision(B)'):.3f}  {f(row, 'metrics/recall(B)'):.3f}  {m50:.3f}  "
                  f"{f(row, 'metrics/mAP50-95(B)'):.3f}   | {best[0]:.3f} ({best[1]:3d})        | "
                  f"{(time.time() - t0) / 3600:4.2f} h", flush=True)
            prev_t = t
        shown = len(rows)
    if done:
        break
    time.sleep(60)
log.close()

print(f"\ntraining process exit code {proc.returncode}, total {(time.time() - t0) / 3600:.2f} h")
if proc.returncode != 0:
    print("Last 30 log lines:\n" + "\n".join(open(LOG).read().splitlines()[-30:]))
    print(f"\nSTOPPED. Checkpoints (if any) are in {RUN}/train/weights; the run can be resumed with --resume.")
else:
    rec = json.load(open(f"{RUN}/run_record.json"))
    vm = rec["val_metrics"]
    print(f"run record : {RUN}/run_record.json")
    print(f"epochs     : {rec['epochs_completed']}   commit {rec['git']['commit'][:7]}   "
          f"dataset {rec['dataset']['version']} {rec['dataset']['fingerprint'][:12]}...")
    print(f"VAL (best.pt, Ultralytics defaults conf=0.001 iou=0.7): P {vm['precision']:.4f}  R {vm['recall']:.4f}  "
          f"mAP50 {vm['map50']:.4f}  mAP50-95 {vm['map50_95']:.4f}")
    print("E001 val, same 324 frames (v1 train only, Colab T4): P 0.7998  R 0.7901  mAP50 0.8527  mAP50-95 0.6115")

E007 started 2026-11-08 01:48 UTC; one line per finished epoch

 epoch | ep time | box loss  cls loss | val P   val R   mAP50  mAP50-95 | best mAP50 (epoch) | elapsed
   1/100 |   1.9 m |   1.298    1.768  | 0.748  0.716  0.793  0.528   | 0.793 (  1)        | 0.07 h
   2/100 |   1.9 m |   1.283    1.167  | 0.744  0.656  0.755  0.495   | 0.793 (  1)        | 0.10 h
   3/100 |   1.8 m |   1.313    1.116  | 0.750  0.641  0.734  0.434   | 0.793 (  1)        | 0.13 h
   4/100 |   1.9 m |   1.282    1.072  | 0.755  0.671  0.758  0.491   | 0.793 (  1)        | 0.17 h
   5/100 |   1.9 m |   1.238    0.994  | 0.775  0.709  0.794  0.529   | 0.794 (  5)        | 0.20 h
   6/100 |   1.9 m |   1.221    0.948  | 0.758  0.714  0.787  0.498   | 0.794 (  5)        | 0.22 h
   7/100 |   1.9 m |   1.192    0.927  | 0.767  0.730  0.805  0.542   | 0.805 (  7)        | 0.25 h
   8/100 |   1.9 m |   1.181    0.917  | 0.792  0.678  0.787  0.540   | 0.805 (  7)        | 0.28 h
   9/100 |   1.9 m |   1.157    0

In [17]:
# Cell 15 - package E007 for download: run folder (weights, run record, curves, plots) + all logs
# Expected time: about 1 min.
# Prints sha256 of the key files: the repo records weights by path + sha256 (CLAUDE.md section 5).

import hashlib, os, zipfile

W = "/kaggle/working"
PKG = f"{W}/E007_package.zip"
SOURCES = [f"{W}/uwct_runs", f"{W}/uwct_logs"]   # E007 + smoke run folders, and every log of this notebook

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

n = 0
with zipfile.ZipFile(PKG, "w", zipfile.ZIP_DEFLATED) as z:
    for src in SOURCES:
        for dirpath, _, files in os.walk(src):
            for name in sorted(files):
                p = os.path.join(dirpath, name)
                z.write(p, os.path.relpath(p, W))
                n += 1

KEY = ["uwct_runs/E007/run_record.json", "uwct_runs/E007/train/results.csv",
       "uwct_runs/E007/train/weights/best.pt", "uwct_runs/E007/train/weights/last.pt"]
for rel in KEY:
    p = f"{W}/{rel}"
    print(f"{rel:40s} {os.path.getsize(p):>12,} B  sha256 {sha(p)}")
print(f"\n{PKG}: {n} files, {os.path.getsize(PKG):,} B  sha256 {sha(PKG)}")
with open(f"{W}/uwct_logs/E007_package_sha256.txt", "w") as fh:
    fh.write(f"{sha(PKG)}  E007_package.zip\n")

uwct_runs/E007/run_record.json                  9,820 B  sha256 b678ded4435195010f935079a9eaeac29e567190bae71b0ad2ef395cc9089a0a
uwct_runs/E007/train/results.csv                7,096 B  sha256 02768a40ea5d80088cbbfbad7ccf36b9a289630e34a2a4149411d2a5551dffd6
uwct_runs/E007/train/weights/best.pt       20,289,541 B  sha256 f5f5a24970026efc9f1fd8034adf4308aa48e16b4f914b8a4fcfe5ea9bdc0f0f
uwct_runs/E007/train/weights/last.pt       20,289,541 B  sha256 9cbaefdf80f8e9894d8af75260cc611cb583a9c631d36e202c13f89fec025971

/kaggle/working/E007_package.zip: 86 files, 86,522,254 B  sha256 8d57c662cb2f72e9145f2274972a50d0eb7baa3917f022fb4f783c9123cda3a3


In [18]:
# Cell 16 - direct download link for the E007 package (the file panel may not refresh)
# Expected time: a few seconds.
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
print("file exists:", os.path.exists("E007_package.zip"), f"({os.path.getsize('E007_package.zip'):,} B)")
display(FileLink("E007_package.zip"))

file exists: True (86,522,254 B)


/kaggle/working/E007_package.zip